In [1]:
# ========================= 
# 1. Import Libraries 
# =========================
import pandas as pd 
import numpy as np 
import torch 
import torch.nn as nn 
import torch.optim as optim 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import r2_score 
from torch.utils.data import TensorDataset, DataLoader

In [4]:
# =========================
# 2. Load Dataset
# ========================= 
df = pd.read_csv("2  powerplant_data.csv")
df.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [6]:
# ========================= 
# 3. Separate Input & Output 
# ========================= 
X = df.drop("PE", axis=1) # input features
y = df["PE"] # target

In [7]:
# =========================
# 4. Train Test Split 
# ========================= 
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42 )

In [8]:
# ========================= 
# 5. Feature Scaling 
# ========================= 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) 
X_test = scaler.transform(X_test)

In [9]:
# ========================= 
# 6. Convert Data into Tensors
# ========================= 
X_train = torch.tensor(X_train, dtype=torch.float32) 
X_test = torch.tensor(X_test, dtype=torch.float32) 
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [10]:
# ========================= 
# 7. Create DataLoader 
# ========================= 
train_data = TensorDataset(X_train, y_train) 
train_loader = DataLoader( train_data, batch_size=32, shuffle=True )

In [21]:
# ========================= 
# 8. Build ANN Model 
# ========================= 
import torch
import torch.nn as nn

class ANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

In [22]:
# =========================
# 9. Create Model 
# ========================= 
model = ANN()

In [23]:
# ========================= 
# 10. Loss Function & Optimizer 
# =========================
criterion = nn.MSELoss() 
optimizer = optim.Adam( model.parameters(), lr=0.001 )

In [24]:
# =========================
# 11. Train Model
# =========================

epochs = 100

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for xb, yb in train_loader:

        # clear old gradients
        optimizer.zero_grad()

        # prediction
        predictions = model(xb)

        # calculate loss
        loss = criterion(predictions, yb)

        # backpropagation
        loss.backward()

        # update weights
        optimizer.step()

        # add batch loss
        total_loss += loss.item()

    # average epoch loss
    avg_loss = total_loss / len(train_loader)

    # print epoch loss
    print(f"Epoch {epoch+1}/{epochs} Loss = {avg_loss}")

Epoch 1/100 Loss = 204288.56868489584
Epoch 2/100 Loss = 173292.22161458334
Epoch 3/100 Loss = 96265.04417317708
Epoch 4/100 Loss = 36024.28837890625
Epoch 5/100 Loss = 19043.398795572917
Epoch 6/100 Loss = 13647.853318277996
Epoch 7/100 Loss = 9743.576407877605
Epoch 8/100 Loss = 6487.10070292155
Epoch 9/100 Loss = 3949.07891998291
Epoch 10/100 Loss = 2294.301369984945
Epoch 11/100 Loss = 1352.9564043680828
Epoch 12/100 Loss = 845.565828704834
Epoch 13/100 Loss = 566.3747644424438
Epoch 14/100 Loss = 402.7355758031209
Epoch 15/100 Loss = 299.0636292457581
Epoch 16/100 Loss = 232.9992262840271
Epoch 17/100 Loss = 187.9431957244873
Epoch 18/100 Loss = 156.15141634941102
Epoch 19/100 Loss = 131.49133273760478
Epoch 20/100 Loss = 112.16064060529074
Epoch 21/100 Loss = 96.52717405954996
Epoch 22/100 Loss = 83.51742713451385
Epoch 23/100 Loss = 73.09228851795197
Epoch 24/100 Loss = 64.38600638707479
Epoch 25/100 Loss = 57.3002681573232
Epoch 26/100 Loss = 51.19297893047333
Epoch 27/100 Loss

In [25]:
# ========================= 
# 12. Test Model
# ========================= 
model.eval()
with torch.no_grad():
    y_pred = model(X_test)

In [26]:
# 13. Evaluate Model 
# ========================= 
mse = criterion(y_pred, y_test)
r2 = r2_score( y_test.numpy(), y_pred.numpy() ) 
print("\nMSE Loss =", mse.item()) 
print("R2 Score =", r2)


MSE Loss = 18.214128494262695
R2 Score = 0.9363462924957275
